# E.R.M.E.S. - Fase 6: Transfer Learning (VGG16) e Risoluzione del Collasso Spaziale

Avendo certificato i limiti strutturali di un'architettura addestrata *from scratch* su un dataset a bassa risoluzione e affetto da forte rumore di fondo (Accuracy bloccata al ~62%), questo notebook implementa il paradigma del **Transfer Learning**.

Sfrutteremo l'architettura **VGG16**, pre-addestrata sul dataset *ImageNet*. L'obiettivo è trasferire la capacità di estrazione gerarchica delle feature spaziali per superare l'ambiguità delle micro-espressioni facciali. Per ovviare al problema del collasso spaziale, causato dai molteplici layer di Pooling su immagini native 48x48, la pipeline integrerà un layer di *Upsampling*.

In [ ]:
"""
Inizializzazione ambiente e raggruppamento delle importazioni.
"""
import os
import logging
import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.layers import Resizing, Flatten, Dense, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard

tf.get_logger().setLevel(logging.ERROR)
plt.style.use('seaborn-v0_8-darkgrid')

print("--- INIZIALIZZAZIONE TRANSFER LEARNING (VGG16) ---")

In [ ]:
"""
Data Engineering: Configurazione pipeline per l'architettura VGG16.
I tensori vengono caricati in RGB (3 canali) e processati con la funzione 
nativa di ImageNet (sottrazione della media dei pixel).
"""
BATCH_SIZE = 64
IMG_SIZE = (48, 48)

train_dir = Path('../data/fer2013/train')
test_dir = Path('../data/fer2013/test')

def process_for_vgg(image, label):
    return preprocess_input(image), label

print("Caricamento Dataset in modalità RGB (3 canali)...")

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir, color_mode='rgb', image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, shuffle=True, label_mode='int'
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    test_dir, color_mode='rgb', image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, shuffle=False, label_mode='int'
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds_raw.map(process_for_vgg, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds_raw.map(process_for_vgg, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Dizionario dei pesi calcolato in fase di EDA
class_weights = {
    0: 1.0266, 1: 9.4066, 2: 1.0010, 3: 0.5684, 
    4: 0.8260, 5: 0.8491, 6: 1.2934
}

print("Pipeline Dati configurata per VGG16.")

## 6.1 Progettazione Architettura Ibrida con Upsampling

L'integrazione della VGG16 segue una logica a tre strati:
1. **Pre-processing Spaziale (Upsampling):** Un layer iniziale di *Resizing* interpola le immagini da 48x48 a 224x224 per impedire la distruzione topologica durante il MaxPooling profondo. 
2. **Base Model (Feature Extractor):** La VGG16 ("congelata") agisce da estrattore matematico di feature visive sfruttando i pesi di ImageNet.
3. **Top Model (Classificatore Custom):** Livelli densi altamente regolarizzati da *Dropout* e *BatchNormalization* per mappare le feature di ImageNet sulle 7 emozioni di FER-2013.

In [ ]:
"""
Costruzione del Modello Ibrido VGG16.
"""
# 1. Base Model (VGG16)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False 

# 2. Architettura Completa
inputs = tf.keras.Input(shape=(48, 48, 3))

# Risoluzione del collasso spaziale tramite stiramento bilineare
x = Resizing(224, 224, interpolation="bilinear")(inputs)

# Passaggio nel Feature Extractor
x = base_model(x, training=False)

# Top Model (Classificatore)
x = Flatten()(x)
x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

outputs = Dense(7, activation='softmax')(x)

vgg_model = Model(inputs, outputs, name="ERMES_VGG16")

vgg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

vgg_model.summary()

In [ ]:
"""
Avvio dell'Addestramento (Fase 1: Feature Extractor Frozen).
"""
log_dir = os.path.join("logs", "fit", "vgg16_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

callbacks_list = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint(filepath='ermes_best_vgg16.h5', monitor='val_loss', save_best_only=True, verbose=1),
    TensorBoard(log_dir=log_dir, histogram_freq=0, write_graph=True)
]

EPOCHS = 100

print("Inizio Addestramento Transfer Learning (VGG16 Frozen) ...\n")
history_vgg = vgg_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks_list
)

In [ ]:
"""
Fase 6.2: Fine-Tuning dell'architettura VGG16.
Scongeliamo l'ultimo blocco convoluzionale per adattare le astrazioni di 
ImageNet al dominio specifico delle espressioni facciali.
"""
print("--- AVVIO FINE-TUNING STRUTTURALE ---")

base_model.trainable = True

# Ricongelamento layer base (Pattern di basso livello: bordi, gradienti)
for layer in base_model.layers[:15]:
    layer.trainable = False

print(f"Layer totali nel Base Model: {len(base_model.layers)}")
print("Layer addestrabili (Scongelati):")
for layer in base_model.layers:
    if layer.trainable:
        print(f" - {layer.name}")

# Ricompilazione con Learning Rate ridotto per non corrompere i pesi preesistenti
vgg_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_finetuning = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(filepath='ermes_best_vgg16_finetuned.h5', monitor='val_loss', save_best_only=True, verbose=1)
]

print("\nRipresa dell'addestramento (Fine-Tuning)...")
history_finetuning = vgg_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    class_weight=class_weights,
    callbacks=callbacks_finetuning
)

In [ ]:
"""
Visualizzazione Storico dell'Addestramento (Training History).
Uniamo i log della Fase 1 (Frozen) e della Fase 2 (Fine-Tuning) per generare 
un unico grafico continuo. Include logica di fallback se il training è stato saltato.
"""
import matplotlib.pyplot as plt
import numpy as np
import os
from IPython.display import SVG, display

plt.style.use('seaborn-v0_8-darkgrid')

def plot_combined_history(hist_vgg, hist_ft):
    # Estrazione metriche
    acc = hist_vgg.history['accuracy']
    val_acc = hist_vgg.history['val_accuracy']
    loss = hist_vgg.history['loss']
    val_loss = hist_vgg.history['val_loss']

    ft_acc = hist_ft.history['accuracy']
    ft_val_acc = hist_ft.history['val_accuracy']
    ft_loss = hist_ft.history['loss']
    ft_val_loss = hist_ft.history['val_loss']

    total_acc = acc + ft_acc
    total_val_acc = val_acc + ft_val_acc
    total_loss = loss + ft_loss
    total_val_loss = val_loss + ft_val_loss

    initial_epochs = len(acc)
    best_total_epoch = initial_epochs + np.argmin(ft_val_loss)

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle('Curve di Apprendimento: Transfer Learning (VGG16 + Fine-Tuning)', fontsize=18, fontweight='bold', y=1.05)

    # Grafico Accuracy
    axes[0].plot(total_acc, label='Training Accuracy', color='#1f77b4', linewidth=2.5)
    axes[0].plot(total_val_acc, label='Validation Accuracy', color='#ff7f0e', linewidth=2.5)
    axes[0].axvline(x=initial_epochs-1, color='gray', linestyle='--', linewidth=2, label='Inizio Fine-Tuning')
    axes[0].axvline(x=best_total_epoch, color='red', linestyle=':', linewidth=2, label=f'Best Epoch ({best_total_epoch+1})')
    axes[0].set_title('Evoluzione dell\'Accuracy', fontsize=15, pad=10)
    axes[0].set_xlabel('Epoche totali', fontsize=12)
    axes[0].set_ylabel('Accuracy', fontsize=12)
    axes[0].legend(loc='lower right', fontsize=11)

    # Grafico Loss
    axes[1].plot(total_loss, label='Training Loss', color='#1f77b4', linewidth=2.5)
    axes[1].plot(total_val_loss, label='Validation Loss', color='#ff7f0e', linewidth=2.5)
    axes[1].axvline(x=initial_epochs-1, color='gray', linestyle='--', linewidth=2, label='Inizio Fine-Tuning')
    axes[1].axvline(x=best_total_epoch, color='red', linestyle=':', linewidth=2, label=f'Best Epoch ({best_total_epoch+1})')
    axes[1].set_title('Evoluzione della Loss', fontsize=15, pad=10)
    axes[1].set_xlabel('Epoche totali', fontsize=12)
    axes[1].set_ylabel('Loss', fontsize=12)
    axes[1].legend(loc='upper right', fontsize=11)

    plt.tight_layout()
    plt.savefig('vgg16_training_history.svg', format='svg', bbox_inches='tight')
    plt.show()

# Logica di Fallback
try:
    plot_combined_history(history_vgg, history_finetuning)
except NameError:
    print("Variabili history non trovate (Addestramento VGG16 saltato)")
    print("Caricamento del grafico vettoriale salvato in precedenza...")
    if os.path.exists('vgg16_training_history.svg'):
        display(SVG(filename='vgg16_training_history.svg'))
    else:
        print("Grafico SVG non trovato. È necessario eseguire il training (Celle 6 e 7)")

In [ ]:
import tensorflow as tf

# --- CARICAMENTO DEL MODELLO ---
try:
    vgg_model = tf.keras.models.load_model('ermes_best_vgg16_finetuned.h5')
    print("Modello NUOVO caricato dalla cartella corrente")
except:
    vgg_model = tf.keras.models.load_model('../models/ermes_best_vgg16_finetuned.h5')
    print("Modello PRE-ESISTENTE caricato dalla cartella '../models/'")

vgg_model.summary()

In [ ]:
"""
Cruscotto di Valutazione Finale: VGG16 Fine-Tuned con Metriche di Validazione Probabilistica e Relativa.
"""
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.metrics import log_loss, cohen_kappa_score, top_k_accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Stile grafico coerente
plt.style.use('seaborn-v0_8-darkgrid')

emotions = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

print("Estrazione predizioni finali dal Test Set (VGG16 Upsampled)...")

y_true = []
y_pred_probs = []

for images, labels in val_ds:
    y_true.extend(labels.numpy())
    preds = vgg_model.predict(images, verbose=0)
    y_pred_probs.extend(preds)

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = np.argmax(y_pred_probs, axis=-1)

# --- Calcolo Metriche di Validazione Probabilistica e Relativa ---
kappa = cohen_kappa_score(y_true, y_pred)
loss_val = log_loss(y_true, y_pred_probs)
top2_acc = top_k_accuracy_score(y_true, y_pred_probs, k=2)

print("\n--- METRICHE DI VALIDAZIONE PROBABILISTICA E RELATIVA (VGG16) ---")
print(f"Log-Loss (Cross-Entropy): {loss_val:.4f}")
print(f"Cohen's Kappa (Accordo netto): {kappa:.4f}")
print(f"Top-2 Accuracy (Emozione reale tra le prime 2 previste): {top2_acc:.4f}\n")

# Calcolo Metriche Base
f1_macro = f1_score(y_true, y_pred, average='macro')
f1_per_class = f1_score(y_true, y_pred, average=None)

print(f"Modello E.R.M.E.S. (VGG16) -> Macro F1-Score Globale: {f1_macro:.4f}\n")
print("--- REPORT DI CLASSIFICAZIONE VGG16 ---")
print(classification_report(y_true, y_pred, target_names=emotions))

# --- Creazione Dashboard Visiva ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [1.5, 1]})
fig.suptitle('Cruscotto di Valutazione Finale - VGG16 (Upsampled + Fine-Tuned)', fontsize=18, fontweight='bold', y=1.05)

# 1. Matrice di Confusione
cm_vgg = confusion_matrix(y_true, y_pred)
sns.heatmap(cm_vgg, annot=True, fmt='d', cmap='Blues', 
            xticklabels=emotions, yticklabels=emotions, annot_kws={"size": 12}, ax=axes[0])
axes[0].set_title('Matrice di Confusione', fontsize=15, pad=15)
axes[0].set_ylabel('Classe Reale', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Classe Predetta', fontsize=12, fontweight='bold')

# 2. Istogramma F1-Score per Classe
sns.barplot(x=f1_per_class, y=emotions, hue=emotions, palette='viridis', legend=False, ax=axes[1], orient='h')
axes[1].set_title('F1-Score per Singola Classe', fontsize=15, pad=15)
axes[1].set_xlabel('F1-Score', fontsize=12, fontweight='bold')
axes[1].set_xlim(0, 1)

# Aggiunta etichette valori sulle barre
for i, v in enumerate(f1_per_class):
    axes[1].text(v + 0.02, i, f"{v:.2f}", color='black', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('vgg16_final_dashboard.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Binarizzazione delle label per il calcolo multi-classe
y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3, 4, 5, 6])
n_classes = y_true_bin.shape[1]

fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Calcolo del Macro-Average AUC
macro_auc = sum(roc_auc.values()) / n_classes
print(f"--- ANALISI ROC/AUC (VGG16) ---")
print(f"Macro-Average AUC: {macro_auc:.4f} (Capacità globale di separazione delle classi)\n")

plt.figure(figsize=(10, 8))
colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink']

for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f"ROC {emotions[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasso di Falsi Positivi (FPR)', fontsize=12, fontweight='bold')
plt.ylabel('Tasso di Veri Positivi (TPR)', fontsize=12, fontweight='bold')
plt.title('Curva ROC Multi-classe (VGG16 Upsampled)', fontsize=15, pad=15)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig('roc_curve_vgg16.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Recupero dinamico dei parametri
vgg_params = vgg_model.count_params() / 1_000_000
ermes_params = 2.5

# Dati per il confronto
modelli = ['E.R.M.E.S. V1\n(CNN Custom)', 'VGG16\n(Upsampled + Fine-Tuned)']
accuracy = [62.0, 62.0]
parametri_milioni = [ermes_params, vgg_params] 

# 2. Calcolo Statistiche di Efficienza
rapporto_dimensione = vgg_params / ermes_params
efficienza_ermes = accuracy[0] / ermes_params      # Accuracy per Milione di parametri
efficienza_vgg = accuracy[1] / vgg_params          # Accuracy per Milione di parametri
rapporto_efficienza = efficienza_ermes / efficienza_vgg

# Stampa a terminale dei risultati ingegneristici
print("--- ANALISI DI EFFICIENZA COMPUTAZIONALE ---")
print(f"Parametri E.R.M.E.S. V1: {ermes_params:.2f} Milioni")
print(f"Parametri VGG16:         {vgg_params:.2f} Milioni")
print(f"Fattore di scala:        La VGG16 è ~{rapporto_dimensione:.0f} volte più pesante.")
print("\nScore di Efficienza (Accuratezza per Milione di Parametri):")
print(f"E.R.M.E.S. V1: {efficienza_ermes:.2f}% / Milione di param.")
print(f"VGG16:         {efficienza_vgg:.2f}% / Milione di param.")
print(f"Conclusione:   E.R.M.E.S. è ~{rapporto_efficienza:.0f} volte più efficiente in termini di costo/guadagno.")
print("-" * 65 + "\n")

# 3. Render Grafico
fig, ax1 = plt.subplots(figsize=(9, 6))
plt.style.use('seaborn-v0_8-darkgrid')

color_acc = '#2ca02c'
color_params = '#d62728'

# Asse Y primario (Accuracy)
ax1.set_xlabel('Architetture Deep Learning', fontsize=12, fontweight='bold')
ax1.set_ylabel('Accuracy (%)', color=color_acc, fontsize=12, fontweight='bold')
bars1 = ax1.bar(np.arange(len(modelli)) - 0.2, accuracy, 0.4, color=color_acc, label='Accuracy')
ax1.tick_params(axis='y', labelcolor=color_acc)
ax1.set_ylim(0, 100)

# Asse Y secondario (Parametri)
ax2 = ax1.twinx()  
ax2.set_ylabel('Milioni di Parametri (M)', color=color_params, fontsize=12, fontweight='bold')
bars2 = ax2.bar(np.arange(len(modelli)) + 0.2, parametri_milioni, 0.4, color=color_params, label='Parametri (M)')
ax2.tick_params(axis='y', labelcolor=color_params)

plt.title('Efficienza Computazionale: E.R.M.E.S. vs VGG16', fontsize=15, pad=15)
plt.xticks(np.arange(len(modelli)), modelli, fontsize=11)

fig.tight_layout()
plt.savefig('efficiency_comparison.svg', format='svg', bbox_inches='tight')
plt.show()

## Conclusioni del Progetto E.R.M.E.S. e Valutazione del Transfer Learning

L'implementazione finale ha fornito un riscontro empirico inequivocabile, certificando i reali limiti del dominio applicativo e la maturità ingegneristica del sistema progettato:

* **Definizione chiara del concetto:** È stato applicato il Transfer Learning accoppiato a un livello di Upsampling Bilineare. Questa tecnica consiste nell'ingrandire artificialmente i tensori di input (da 48x48 a 224x224 pixel) prima di sottoporli all'architettura VGG16 e procedere con lo scongelamento (Fine-Tuning) degli strati convoluzionali più profondi.
* **Motivazione:** Il passaggio alla VGG16 nasce dal tentativo di superare l'asintoto predittivo del 62% imposto dalla limitata profondità della nostra CNN Custom. L'Upsampling si è reso strettamente necessario per evitare il collasso spaziale: i molteplici livelli di MaxPooling della VGG16 distruggevano la griglia nativa 48x48 riducendola a 1.5x1.5 pixel, rendendo impossibile per la rete calcolare le distanze facciali geometriche.
* **Dettagli implementativi o strutturali:** La rete esegue internamente il resizing sui tensori GPU-side. I pesi pre-addestrati di ImageNet sono stati inizialmente congelati per addestrare un classificatore denso finale. Successivamente, il blocco convoluzionale 5 è stato scongelato per allineare l'estrazione semantica al dominio delle espressioni facciali, applicando un learning rate estremamente ridotto (1e-5) per non corrompere i gradienti pre-esistenti.
* **Implicazioni pratiche e applicazioni:** L'analisi delle metriche probabilistiche e strutturali smonta l'approccio Model-Centric in contesti di forte rumore, introducendo il concetto fondamentale di 'Model Capacity': 
    * **Capacità del Modello vs Qualità del Dato:** Nonostante la VGG16 sia un colosso da 27.7 milioni di parametri (11 volte più pesante rispetto ai 2.5 milioni della nostra CNN), ha registrato lo stesso limite di Accuracy (62%) e di confidenza (Top-2 Accuracy al ~79%, Macro-AUC a 0.88). Questo accade perché l'elevata capacità della VGG16 è 'affamata' di informazioni (es. micro-rughe d'espressione) che nella bassa risoluzione del dataset FER-2013 non esistono fisicamente.
    * **Efficienza Computazionale:** Il calcolo del rapporto costo/guadagno dimostra che l'architettura E.R.M.E.S. V1 genera uno score del 24.80% di Accuratezza per ogni milione di parametri, contro il 2.24% della VGG16. La nostra rete custom è quindi ~11 volte più efficiente.

**Conclusione Finale:** Lo schianto di un'architettura State of the Art contro il medesimo Upper Bound della nostra rete custom dimostra che il collo di bottiglia attuale è il limite intrinseco di informazione pulita presente nel dataset. 
Se in un'applicazione futura venissero iniettati dati perfetti (ad altissima risoluzione nativa, privi di rumore e con Face Alignment perfetto), la dinamica si invertirebbe, la VGG16 sfrutterebbe la sua immensa Model Capacity per isolare micro-dettagli preclusi alla CNN custom, superandola nettamente in Accuratezza. Tuttavia, in uno scenario ingegneristico reale (es. analisi in real-time su webcam e dispositivi Edge a bassa potenza), la VGG16 risulterebbe ingiustificatamente lenta e dispendiosa. E.R.M.E.S. V1 si conferma la soluzione ottimale, bilanciando sapientemente il massimo potere estrattivo oggi consentito dai dati con un'impronta computazionale efficiente.